# Assignment 2

### Part D Fine Tuning Model

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Load parquet file
import pandas as pd
file_path = '/content/drive/MyDrive/Colab Notebooks/patents_50k_green.parquet'

df = pd.read_parquet(file_path)

In [3]:
print(df.head())
print(df['is_green_silver'].value_counts())

        id        date                                               text  \
0  8929789  2015-01-06  1. A fixing device comprising: a rotatable, en...   
1  9253996  2016-02-09  1. A method of recovering pectin from citrus p...   
2  9580824  2017-02-28  1. An anion-conducting polymeric membrane comp...   
3  9383068  2016-07-05  1. An LED based lighting system, comprising: a...   
4  8915542  2014-12-23  1. A sunroof apparatus comprising: a movable r...   

   Y02A  Y02B  Y02C  Y02D  Y02E  Y02P  Y02T  Y02W  is_green_silver  
0     0     0     0     0     0     0     0     0                0  
1     0     0     0     0     1     1     0     0                1  
2     0     0     0     0     1     0     0     0                1  
3     0     1     0     0     0     0     0     0                1  
4     0     0     0     0     0     0     0     0                0  
is_green_silver
0    25000
1    25000
Name: count, dtype: int64


In [4]:
# Shuffle
df_shuffled = df.sample(frac=1, random_state=42)

# Split
train_silver = df_shuffled.iloc[:40000]
eval_silver = df_shuffled.iloc[40000:45000]
pool_unlabeled = df_shuffled.iloc[45000:]

print(f"Train: {len(train_silver)}, Eval: {len(eval_silver)}, Pool: {len(pool_unlabeled)}")


Train: 40000, Eval: 5000, Pool: 5000


In [5]:
train_silver.to_csv("train_silver.csv", index=False)
eval_silver.to_csv("eval_silver.csv", index=False)
print("Saved!")

Saved!


In [6]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('AI-Growth-Lab/PatentSBERTa', device='cuda')



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import torch
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# --------------------------
# 1️⃣ Load your data
# --------------------------
# Load the HITL gold labels you created in VS Code
hitl_gold = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/hitl_gold_100.csv')


print(f"Train silver: {len(train_silver)} samples")
print(f"Eval silver: {len(eval_silver)} samples")
print(f"HITL gold: {len(hitl_gold)} samples")

# --------------------------
# 2️⃣ Merge gold labels into training set
# --------------------------
# Prepare train_silver with label column
train_silver = train_silver.copy()
train_silver['label'] = train_silver['is_green_silver']

# Prepare gold_100 with label column
gold_100 = hitl_gold.copy()
gold_100['label'] = gold_100['is_green_human']

# Combine: train_silver + gold_100
train_combined = pd.concat([
    train_silver[['text', 'label']],
    gold_100[['text', 'label']]
], ignore_index=True)

print(f"\nCombined training set: {len(train_combined)} samples")
print(f"  - Train silver: {len(train_silver)}")
print(f"  - Gold labels: {len(gold_100)}")

# Prepare eval set
eval_silver = eval_silver.copy()
eval_silver['label'] = eval_silver['is_green_silver']

# --------------------------
# 3️⃣ Convert to HuggingFace Dataset
# --------------------------
train_ds = Dataset.from_pandas(train_combined[['text', 'label']])
eval_ds = Dataset.from_pandas(eval_silver[['text', 'label']])
gold_ds = Dataset.from_pandas(gold_100[['text', 'label']])

dataset_dict = DatasetDict({
    "train": train_ds,
    "eval": eval_ds
})

print("\n✅ Datasets created")

# --------------------------
# 4️⃣ Tokenizer
# --------------------------
MODEL_NAME = "AI-Growth-Lab/PatentSBERTa"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch['text'],
        padding='max_length',
        truncation=True,
        max_length=256
    )

print("Tokenizing datasets...")
tokenized_datasets = dataset_dict.map(tokenize, batched=True)
tokenized_datasets.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

# Tokenize gold set for evaluation
gold_tokenized = gold_ds.map(tokenize, batched=True)
gold_tokenized.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

print("✅ Tokenization complete")

# --------------------------
# 5️⃣ Load model
# --------------------------
print("\nLoading PatentSBERTa model...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True  # Add this to avoid warnings
)
print("✅ Model loaded")

# --------------------------
# 6️⃣ Define metrics
# --------------------------
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary'
    )
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

# --------------------------
# 7️⃣ Training arguments
# --------------------------
training_args = TrainingArguments(
    output_dir="./patentsbert_finetune",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    eval_strategy="epoch",  # Changed from evaluation_strategy
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none"
)
# --------------------------
# 8️⃣ Trainer
# --------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['eval'],
    compute_metrics=compute_metrics
)

# --------------------------
# 9️⃣ Train
# --------------------------
print("\n" + "="*80)
print("🚀 STARTING FINE-TUNING")
print("="*80)

trainer.train()

print("\n✅ Training complete!")

# --------------------------
# 🔟 Evaluate
# --------------------------
print("\n" + "="*80)
print("📊 EVALUATION RESULTS")
print("="*80)

print("\n1️⃣ Evaluation on eval_silver (5,000 samples):")
eval_metrics = trainer.evaluate(tokenized_datasets['eval'])
for key, value in eval_metrics.items():
    print(f"  {key}: {value:.4f}")

print("\n2️⃣ Evaluation on gold_100 (HITL labeled):")
gold_metrics = trainer.evaluate(gold_tokenized)
for key, value in gold_metrics.items():
    print(f"  {key}: {value:.4f}")

# --------------------------
# 💾 Save the fine-tuned model
# --------------------------
model.save_pretrained("./patentsbert_finetuned_final")
tokenizer.save_pretrained("./patentsbert_finetuned_final")
print("\n✅ Model saved to './patentsbert_finetuned_final'")

# --------------------------
# 📈 Summary
# --------------------------
print("\n" + "="*80)
print("✅ FINE-TUNING COMPLETE!")
print("="*80)
print(f"Training set size: {len(train_combined)}")
print(f"Eval set size: {len(eval_silver)}")
print(f"Gold set size: {len(gold_100)}")
print("\nResults:")
print(f"  Eval F1: {eval_metrics['eval_f1']:.4f}")
print(f"  Gold F1: {gold_metrics['eval_f1']:.4f}")

Train silver: 40000 samples
Eval silver: 5000 samples
HITL gold: 100 samples

Combined training set: 40100 samples
  - Train silver: 40000
  - Gold labels: 100

✅ Datasets created
Tokenizing datasets...


Map:   0%|          | 0/40100 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

✅ Tokenization complete

Loading PatentSBERTa model...


...

There were missing keys in the checkpoint model loaded: ['mpnet.embeddings.LayerNorm.weight', 'mpnet.embeddings.LayerNorm.bias', 'mpnet.encoder.layer.0.attention.LayerNorm.weight', 'mpnet.encoder.layer.0.attention.LayerNorm.bias', 'mpnet.encoder.layer.0.output.LayerNorm.weight', 'mpnet.encoder.layer.0.output.LayerNorm.bias', 'mpnet.encoder.layer.1.attention.LayerNorm.weight', 'mpnet.encoder.layer.1.attention.LayerNorm.bias', 'mpnet.encoder.layer.1.output.LayerNorm.weight', 'mpnet.encoder.layer.1.output.LayerNorm.bias', 'mpnet.encoder.layer.2.attention.LayerNorm.weight', 'mpnet.encoder.layer.2.attention.LayerNorm.bias', 'mpnet.encoder.layer.2.output.LayerNorm.weight', 'mpnet.encoder.layer.2.output.LayerNorm.bias', 'mpnet.encoder.layer.3.attention.LayerNorm.weight', 'mpnet.encoder.layer.3.attention.LayerNorm.bias', 'mpnet.encoder.layer.3.output.LayerNorm.weight', 'mpnet.encoder.layer.3.output.LayerNorm.bias', 'mpnet.encoder.layer.4.attention.LayerNorm.weight', 'mpnet.encoder.layer.4.atte


✅ Training complete!

📊 EVALUATION RESULTS

1️⃣ Evaluation on eval_silver (5,000 samples):


  eval_loss: 0.4001
  eval_accuracy: 0.8172
  eval_precision: 0.8246
  eval_recall: 0.8115
  eval_f1: 0.8180
  eval_runtime: 29.1117
  eval_samples_per_second: 171.7520
  eval_steps_per_second: 10.7520
  epoch: 1.0000

2️⃣ Evaluation on gold_100 (HITL labeled):
  eval_loss: 0.7173
  eval_accuracy: 0.5400
  eval_precision: 0.9388
  eval_recall: 0.5169
  eval_f1: 0.6667
  eval_runtime: 0.6235
  eval_samples_per_second: 160.3900
  eval_steps_per_second: 11.2270
  epoch: 1.0000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Model saved to './patentsbert_finetuned_final'

✅ FINE-TUNING COMPLETE!
Training set size: 40100
Eval set size: 5000
Gold set size: 100

Results:
  Eval F1: 0.8180
  Gold F1: 0.6667


In [8]:
!zip -r patentsbert_finetuned_final.zip patentsbert_finetuned_final

# Check the size
!ls -lh patentsbert_finetuned_final.zip


  adding: patentsbert_finetuned_final/ (stored 0%)
  adding: patentsbert_finetuned_final/config.json (deflated 49%)
  adding: patentsbert_finetuned_final/tokenizer.json (deflated 71%)
  adding: patentsbert_finetuned_final/tokenizer_config.json (deflated 48%)
  adding: patentsbert_finetuned_final/model.safetensors (deflated 8%)
-rw-r--r-- 1 root root 387M Feb 25 14:27 patentsbert_finetuned_final.zip


In [9]:
from huggingface_hub import notebook_login
notebook_login()

In [10]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Load your saved model
model = AutoModelForSequenceClassification.from_pretrained('./patentsbert_finetuned_final')
tokenizer = AutoTokenizer.from_pretrained('./patentsbert_finetuned_final')

# Push to Hugging Face (replace 'your-username' with your actual HF username)
model_name = "Anders-sonderby/patentsbert-finetune_1"
model.push_to_hub(model_name)
tokenizer.push_to_hub(model_name)

print(f"✅ Model uploaded to https://huggingface.co/{model_name}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...1otanll/model.safetensors:   0%|          |  562kB /  438MB            

README.md: 0.00B [00:00, ?B/s]

✅ Model uploaded to https://huggingface.co/Anders-sonderby/patentsbert-finetune_1


In [ ]:
# Recreate datasets without pandas index
train_ds = Dataset.from_pandas(train_combined[['text', 'label']], preserve_index=False)
eval_ds = Dataset.from_pandas(eval_silver[['text', 'label']], preserve_index=False)
gold_ds = Dataset.from_pandas(gold_100[['text', 'label']], preserve_index=False)

# Create DatasetDict and push
dataset_dict = DatasetDict({
    'train': train_ds,
    'eval': eval_ds,
    'gold_test': gold_ds
})

dataset_dict.push_to_hub("Anders-sonderby/patent-green-classification")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/41 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :   3%|3         |  526kB / 16.3MB            

...

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 44.5kB / 44.5kB            

CommitInfo(commit_url='https://huggingface.co/datasets/Anders-sonderby/patent-green-classification/commit/3980b11e4936198d00a8a4e7102968d1e5e2a7aa', commit_message='Upload dataset', commit_description='', oid='3980b11e4936198d00a8a4e7102968d1e5e2a7aa', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Anders-sonderby/patent-green-classification', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Anders-sonderby/patent-green-classification'), pr_revision=None, pr_num=None)